# Lezione 10 — Dashboard interattive con Dash: callback e controlli

Nella lezione precedente hai costruito dashboard con Dash, ma erano **fotografie costose**: apparivano bene, ma nessuno poteva toccare nulla. Oggi le rendi interattive con i **callback**: il meccanismo che collega "l'utente ha fatto qualcosa" con "il grafico si aggiorna automaticamente".


## ⚠️ Prima di iniziare: Dash non funziona nativamente su Google Colab

Come nella lezione precedente, Dash avvia un proprio server web (Flask) che ascolta su una porta locale (`127.0.0.1:8050`). Colab esegue il tuo notebook su una macchina remota a cui non hai accesso diretto per quella via, quindi **il pattern `app.run()` normale non ti aprirà la dashboard dentro Colab**.

Hai due opzioni per lavorare con gli esempi di questa lezione:

### Opzione A — File `.py` nel tuo terminale o IDE (consigliata per esercitarti "sul serio")

1. Copia il codice dell'esempio in un file `.py` (in questo notebook lo generiamo con la magic `%%writefile`).
2. Esegui quel file da un terminale locale (non in Colab): `python nome_file.py`.
3. Apri nel tuo browser l'URL che appare nel terminale (normalmente `http://127.0.0.1:8050/`).
4. Cambia i controlli e osserva come il callback aggiorna il grafico in diretta.

### Opzione B — `app.run(jupyter_mode='inline')` (per vedere il risultato senza uscire dal notebook)

Da Dash 2.11, `dash.Dash` sa mostrare l'app **incorporata dentro una cella di output** del notebook (funziona in Colab e in Jupyter classico) senza installare nulla di aggiuntivo. L'unica differenza rispetto all'Opzione A è l'ultima riga:

```python
app = dash.Dash(__name__)
app.layout = ...      # come sempre
# ... callback come sempre ...

app.run(jupyter_mode='inline')   # <- la differenza chiave rispetto ad app.run()
```

In questo notebook ti diamo **entrambe le varianti** per l'esempio principale (callback semplice di 1 Input/1 Output), così scegli quella che funziona meglio per te. Per gli esempi successivi seguiamo lo stesso pattern.

## 1. L'analogia: il callback come un cameriere

Immagina un ristorante. Tu, seduto al tavolo, ordini un piatto — questo è l'**Input**: il valore che hai appena inserito o selezionato (scrivi un anno, scegli una regione, muovi uno slider...).

Il cameriere non cucina al tuo tavolo: va in cucina, dove qualcuno prepara il piatto secondo il tuo ordine — questa è **la funzione** che scrivi tu. Quando è pronto, il cameriere torna e te lo porta — questo è l'**Output**: ciò che si aggiorna sullo schermo (un grafico, un testo, se un controllo resta disabilitato...).

Tu non dici mai al cameriere "aggiorna il piatto" manualmente. Il cameriere è sempre attento: appena rileva che hai ordinato qualcosa (hai cambiato l'Input), scatena automaticamente tutto il processo. Questo è esattamente ciò che fa il decoratore `@app.callback`.

Senza callback, Dash è solo un modo elegante di mostrare un'immagine fissa. Con i callback, è un'applicazione reale.


## 2. L'anatomia di un callback

Un callback in Dash ha sempre questa forma:

```python
@app.callback(Output(component_id='...', component_property='...'),
              Input(component_id='...', component_property='...'))
def mi_funcion(valor_del_input):
    # ... calcolare qualcosa con valor_del_input ...
    return resultado
```

Tre elementi che vanno sempre insieme:

- **`Output`**: dice a Dash *quale componente* della pagina verrà aggiornato e *quale proprietà* di quel componente cambia.
- **`Input`**: dice a Dash *quale componente* sorvegliare e *quale proprietà* di quel componente, al cambiare, deve scatenare la funzione.
- **La funzione decorata**: riceve come argomento il valore attuale dell'Input (nello stesso ordine in cui hai dichiarato gli Input) e deve `return`are il nuovo valore per l'Output.

`component_id` e `component_property` identificano un componente senza ambiguità:

- `component_id`: l'identificatore unico che hai assegnato al componente nel layout (`id=...`).
- `component_property`: quale attributo di quel componente leggi o scrivi. I più comuni:
  - `value` — il valore attuale di un `dcc.Input`, `dcc.Dropdown` o `dcc.RadioItems` (tipico per gli **Input**).
  - `figure` — la figura Plotly mostrata da un `dcc.Graph` (tipico per gli **Output**).
  - `children` — il contenuto interno di un `html.Div` (utile quando inserisci un `dcc.Graph` completo dentro una divisione vuota).
  - `disabled` — se un controllo è disabilitato o no.

**Regola d'oro dell'ordine:** l'ordine della lista di `Output` deve coincidere con l'ordine del `return`; l'ordine della lista di `Input` deve coincidere con l'ordine degli argomenti della funzione.


## 3. Esempio 1 — Un Input, un Output

Il caso più semplice: un campo numerico dove l'utente scrive un **anno**, e un grafico a linee che ricalcola il **ritardo medio di arrivo per mese** per quell'anno.

Usiamo il dataset `airline_data.csv` (Airline Reporting Carrier On-Time Performance), lo stesso che già conosci dalle lezioni precedenti.

Per prima cosa, prepariamo i dati (questa cella funziona su Colab senza problemi — è solo pandas):


In [ ]:
import pandas as pd
import plotly.graph_objs as go
import plotly.express as px

# Leggiamo il dataset delle compagnie aeree (lo stesso delle lezioni precedenti)
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

airline_data.head()


### Opzione A — Salvarlo come `.py` ed eseguirlo in un terminale

Esegui la cella seguente: crea un file `clase10_ejemplo1_callback_simple.py` con l'app completa. Poi, da un terminale (fuori da Colab), esegui:

```
python clase10_ejemplo1_callback_simple.py
```

e apri l'URL indicato (normalmente `http://127.0.0.1:8050/`).


In [ ]:
%%writefile clase10_ejemplo1_callback_simple.py
# Import required libraries
import pandas as pd
import plotly.graph_objs as go
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

# Read the airline data into pandas dataframe
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

# Create a dash application
app = dash.Dash(__name__)

# Layout: titolo, campo di input per l'anno, e un dcc.Graph VUOTO
# (la figura la riempirà il callback, non si passa figure= qui)
app.layout = html.Div(children=[
    html.H1('Airline Performance Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
              type='number', style={'height': '50px', 'font-size': 35}), ],
              style={'font-size': 40}),
    html.Br(),
    html.Br(),
    html.Div(dcc.Graph(id='line-plot')),
])


# Callback: quando cambia il "value" di input-year, si attiva get_graph()
# e il suo risultato diventa il "figure" di line-plot.
@app.callback(Output(component_id='line-plot', component_property='figure'),
              Input(component_id='input-year', component_property='value'))
def get_graph(entered_year):
    # Select data based on the entered year
    df = airline_data[airline_data['Year'] == int(entered_year)]

    # Group the data by Month and compute the average over arrival delay time.
    line_data = df.groupby('Month')['ArrDelay'].mean().reset_index()

    fig = go.Figure(data=go.Scatter(x=line_data['Month'], y=line_data['ArrDelay'],
                                     mode='lines', marker=dict(color='green')))
    fig.update_layout(title='Month vs Average Flight Delay Time',
                       xaxis_title='Month', yaxis_title='ArrDelay')
    return fig


# Run the app
if __name__ == '__main__':
    app.run(debug=True)


### Opzione B — `app.run(jupyter_mode='inline')` (vederlo dentro il notebook)

Stesso layout e stesso callback, cambiando solo l'ultima riga in `app.run(jupyter_mode='inline')` invece di `app.run()`.

In [ ]:
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objs as go

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Airline Performance Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
              type='number', style={'height': '50px', 'font-size': 35}), ],
              style={'font-size': 40}),
    html.Br(),
    html.Br(),
    html.Div(dcc.Graph(id='line-plot')),
])


@app.callback(Output(component_id='line-plot', component_property='figure'),
              Input(component_id='input-year', component_property='value'))
def get_graph(entered_year):
    df = airline_data[airline_data['Year'] == int(entered_year)]
    line_data = df.groupby('Month')['ArrDelay'].mean().reset_index()

    fig = go.Figure(data=go.Scatter(x=line_data['Month'], y=line_data['ArrDelay'],
                                     mode='lines', marker=dict(color='green')))
    fig.update_layout(title='Month vs Average Flight Delay Time',
                       xaxis_title='Month', yaxis_title='ArrDelay')
    return fig


# mode='inline' -> l'app appare incorporata come output di questa cella
app.run(jupyter_mode='inline')


Leggilo come una frase: "quando il `value` del componente `input-year` cambia, chiama `get_graph` con quel valore, e il risultato restituito diventa il nuovo `figure` del componente `line-plot`". La figura si costruisce con `plotly.graph_objects` (`go.Figure` / `go.Scatter`), altrettanto valido di Plotly Express.


## 4. Esempio 2 — Un Input, più Output

Un callback non è limitato a un solo Output. Possiamo usare lo stesso campo dell'anno come unico Input, ma aggiornare **più** grafici contemporaneamente: ritardi per vettore, clima, sistema aereo nazionale (NAS), sicurezza e aeromobile in ritardo. Per questo, sia gli `Output` sia il valore di `return` si raggruppano in **liste**, nello stesso ordine.


In [ ]:
import plotly.express as px

def compute_info(airline_data, entered_year):
    """Separiamo il calcolo pesante (raggruppare/mediare) dalla costruzione dei grafici."""
    df = airline_data[airline_data['Year'] == int(entered_year)]
    avg_car = df.groupby(['Month', 'Reporting_Airline'])['CarrierDelay'].mean().reset_index()
    avg_weather = df.groupby(['Month', 'Reporting_Airline'])['WeatherDelay'].mean().reset_index()
    avg_NAS = df.groupby(['Month', 'Reporting_Airline'])['NASDelay'].mean().reset_index()
    avg_sec = df.groupby(['Month', 'Reporting_Airline'])['SecurityDelay'].mean().reset_index()
    avg_late = df.groupby(['Month', 'Reporting_Airline'])['LateAircraftDelay'].mean().reset_index()
    return avg_car, avg_weather, avg_NAS, avg_sec, avg_late


# Così si dichiara il callback con 5 Output e 1 Input (non lo eseguiamo qui,
# è lo stesso pattern di Opzione A / Opzione B che già conosci):
#
# @app.callback([
#                Output(component_id='carrier-plot', component_property='figure'),
#                Output(component_id='weather-plot', component_property='figure'),
#                Output(component_id='nas-plot', component_property='figure'),
#                Output(component_id='security-plot', component_property='figure'),
#                Output(component_id='late-plot', component_property='figure')
#                ],
#                Input(component_id='input-year', component_property='value'))
# def get_graph(entered_year):
#     avg_car, avg_weather, avg_NAS, avg_sec, avg_late = compute_info(airline_data, entered_year)
#
#     carrier_fig = px.line(avg_car, x='Month', y='CarrierDelay', color='Reporting_Airline',
#                            title='Average carrier delay time (minutes) by airline')
#     weather_fig = px.line(avg_weather, x='Month', y='WeatherDelay', color='Reporting_Airline',
#                            title='Average weather delay time (minutes) by airline')
#     nas_fig = px.line(avg_NAS, x='Month', y='NASDelay', color='Reporting_Airline',
#                        title='Average NAS delay time (minutes) by airline')
#     sec_fig = px.line(avg_sec, x='Month', y='SecurityDelay', color='Reporting_Airline',
#                        title='Average security delay time (minutes) by airline')
#     late_fig = px.line(avg_late, x='Month', y='LateAircraftDelay', color='Reporting_Airline',
#                         title='Average late aircraft delay time (minutes) by airline')
#
#     # L'ORDINE del return deve coincidere con l'ordine degli Output sopra
#     return [carrier_fig, weather_fig, nas_fig, sec_fig, late_fig]

# Proviamo comunque la funzione ausiliaria per confermare che i dati escano bene:
avg_car, avg_weather, avg_NAS, avg_sec, avg_late = compute_info(airline_data, 2010)
avg_car.head()


La regola è semplice: l'**ordine** della lista di `Output` deve coincidere esattamente con l'**ordine** della lista che restituisci nel `return`. Qui appare anche un pattern utile: separare il calcolo pesante (`compute_info`) dalla funzione del callback, che si occupa solo di "costruire i grafici".


## 5. Esempio 3 — Due controlli che alimentano lo stesso callback

Un callback può anche dipendere da **più** controlli contemporaneamente. Questa è la dashboard degli incendi in Australia: un `dcc.RadioItems` per scegliere la regione e un `dcc.Dropdown` per scegliere l'anno, entrambi che alimentano la stessa coppia di grafici (un **pie chart** e un **bar chart**).

Dataset: incendi boschivi in Australia.


In [ ]:
import pandas as pd

# Leggiamo il dataset degli incendi in Australia
df = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv')

# Estraiamo anno e mese dalla colonna Date
df['Month'] = pd.to_datetime(df['Date']).dt.month_name()
df['Year'] = pd.to_datetime(df['Date']).dt.year

df.head()


Nota due dettagli chiave del layout di questo esempio:

- Le divisioni di output sono **vuote** (`html.Div([], id='plot1')`) — non hanno un `dcc.Graph` predefinito. Il callback inserirà il grafico completo al loro interno, per questo il `component_property` dell'Output è `children` e non `figure`.
- La funzione del callback riceve **due** argomenti (`input_region`, `input_year`), nello stesso ordine in cui sono stati dichiarati i due `Input`. Non importa quale dei due controlli cambi l'utente — il callback si attiva e riceve i valori attuali di entrambi.

### Opzione A — Salvarlo come `.py` ed eseguirlo in un terminale


In [ ]:
%%writefile clase10_ejemplo3_region_year.py
# Import required libraries
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

# Read the wildfire data into pandas dataframe
df = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Historical_Wildfires.csv')

# Extract year and month from the date column
df['Month'] = pd.to_datetime(df['Date']).dt.month_name()
df['Year'] = pd.to_datetime(df['Date']).dt.year

# Create a dash application
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Australia Wildfire Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 26}),

    # Outer division con due divisioni interne: controlli e grafici
    html.Div([
        html.Div([
            html.H2('Select Region:', style={'margin-right': '2em'}),
            dcc.RadioItems([{"label": "New South Wales", "value": "NSW"},
                            {"label": "Northern Territory", "value": "NT"},
                            {"label": "Queensland", "value": "QL"},
                            {"label": "South Australia", "value": "SA"},
                            {"label": "Tasmania", "value": "TA"},
                            {"label": "Victoria", "value": "VI"},
                            {"label": "Western Australia", "value": "WA"}],
                           "NSW", id='region', inline=True)]),

        html.Div([
            html.H2('Select Year:', style={'margin-right': '2em'}),
            dcc.Dropdown(df.Year.unique(), value=2005, id='year')
        ]),

        # Divisioni vuote: qui il callback inserirà i dcc.Graph completi
        html.Div([
            html.Div([], id='plot1'),
            html.Div([], id='plot2')
        ], style={'display': 'flex'}),
    ])
])


@app.callback([Output(component_id='plot1', component_property='children'),
               Output(component_id='plot2', component_property='children')],
              [Input(component_id='region', component_property='value'),
               Input(component_id='year', component_property='value')])
def reg_year_display(input_region, input_year):
    # data
    region_data = df[df['Region'] == input_region]
    y_r_data = region_data[region_data['Year'] == input_year]

    # Plot one - Monthly Average Estimated Fire Area
    est_data = y_r_data.groupby('Month')['Estimated_fire_area'].mean().reset_index()
    fig1 = px.pie(est_data, values='Estimated_fire_area', names='Month',
                  title="{} : Monthly Average Estimated Fire Area in year {}".format(input_region, input_year))

    # Plot two - Monthly Average Count of Pixels for Presumed Vegetation Fires
    veg_data = y_r_data.groupby('Month')['Count'].mean().reset_index()
    fig2 = px.bar(veg_data, x='Month', y='Count',
                  title='{} : Average Count of Pixels for Presumed Vegetation Fires in year {}'.format(input_region, input_year))

    return [dcc.Graph(figure=fig1),
            dcc.Graph(figure=fig2)]


if __name__ == '__main__':
    app.run(debug=True)


### Opzione B — `app.run(jupyter_mode='inline')`

In [ ]:
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Australia Wildfire Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 26}),

    html.Div([
        html.Div([
            html.H2('Select Region:', style={'margin-right': '2em'}),
            dcc.RadioItems([{"label": "New South Wales", "value": "NSW"},
                            {"label": "Northern Territory", "value": "NT"},
                            {"label": "Queensland", "value": "QL"},
                            {"label": "South Australia", "value": "SA"},
                            {"label": "Tasmania", "value": "TA"},
                            {"label": "Victoria", "value": "VI"},
                            {"label": "Western Australia", "value": "WA"}],
                           "NSW", id='region', inline=True)]),

        html.Div([
            html.H2('Select Year:', style={'margin-right': '2em'}),
            dcc.Dropdown(df.Year.unique(), value=2005, id='year')
        ]),

        html.Div([
            html.Div([], id='plot1'),
            html.Div([], id='plot2')
        ], style={'display': 'flex'}),
    ])
])


@app.callback([Output(component_id='plot1', component_property='children'),
               Output(component_id='plot2', component_property='children')],
              [Input(component_id='region', component_property='value'),
               Input(component_id='year', component_property='value')])
def reg_year_display(input_region, input_year):
    region_data = df[df['Region'] == input_region]
    y_r_data = region_data[region_data['Year'] == input_year]

    est_data = y_r_data.groupby('Month')['Estimated_fire_area'].mean().reset_index()
    fig1 = px.pie(est_data, values='Estimated_fire_area', names='Month',
                  title="{} : Monthly Average Estimated Fire Area in year {}".format(input_region, input_year))

    veg_data = y_r_data.groupby('Month')['Count'].mean().reset_index()
    fig2 = px.bar(veg_data, x='Month', y='Count',
                  title='{} : Average Count of Pixels for Presumed Vegetation Fires in year {}'.format(input_region, input_year))

    return [dcc.Graph(figure=fig1),
            dcc.Graph(figure=fig2)]


app.run(jupyter_mode='inline')


## Punti chiave

- Un callback collega un `Input` (ciò che l'utente cambia) con un `Output` (ciò che si aggiorna) tramite una funzione che scrivi tu.
- `component_id` identifica il componente; `component_property` identifica quale attributo viene letto (`value` normalmente) o scritto (`figure`, `children`, `disabled`, tra gli altri).
- L'ordine conta: l'ordine degli `Output` nella lista deve coincidere con l'ordine dei valori nel `return`; l'ordine degli `Input` deve coincidere con l'ordine degli argomenti della funzione.
- Un callback può avere più Output (Esempio 2) e anche più Input contemporaneamente (Esempio 3) — entrambi i pattern usano semplicemente delle liste.
- Se l'Output punta a `figure`, restituisci direttamente la figura Plotly. Se punta a `children` di una divisione vuota, restituisci il componente completo (ad esempio `dcc.Graph(figure=fig)`).
- Dash non funziona nativamente su Colab: usa un file `.py` in terminale/IDE, oppure `app.run(jupyter_mode='inline')` dentro il notebook (senza installare nulla di aggiuntivo da Dash 2.11).

Ora vai al notebook di pratica (`clase10_practica_dash-interactividad_IT.ipynb`) per applicare questi pattern tu stesso.